In [60]:
# COMPLETE LSTM EXPERIMENT
#
# X Task: Sentiment Classification
# Y Task: Spam Classification
#
# 1. Train X model on X task
# 2. Train Y model on Y task
# 3. Fine-tune X model on Y task
# 4. Compare Y model output with fine-tuned X model output

import os
import pickle
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



np.random.seed(42)
tf.random.set_seed(42)


#  X TASK DATASET: SENTIMENT CLASSIFICATION


x_task_texts = [
    "I love this movie",
    "This film was amazing",
    "The story was wonderful",
    "The acting was excellent",
    "I enjoyed the movie very much",
    "This movie was fantastic",
    "The direction was brilliant",
    "The film made me happy",
    "It was a beautiful story",
    "The performance was great",
    "I really liked this film",
    "This was a pleasant experience",
    "The movie was inspiring",
    "The characters were lovable",
    "This film is one of the best",

    "I hate this movie",
    "This film was terrible",
    "The story was boring",
    "The acting was bad",
    "I did not enjoy the movie",
    "This movie was awful",
    "The direction was poor",
    "The film made me angry",
    "It was a horrible story",
    "The performance was weak",
    "I really disliked this film",
    "This was a bad experience",
    "The movie was disappointing",
    "The characters were annoying",
    "This film is one of the worst"
]

x_task_labels = np.array([
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0
])


# Y TASK DATASET: SPAM CLASSIFICATION
# 1 = Spam, 0 = Not Spam


y_task_texts = [
    "Congratulations you won a free prize",
    "Claim your lottery reward now",
    "You have won free money",
    "Click here to get a free gift",
    "Urgent you won a cash prize",
    "Get free coupons now",
    "Win money by clicking this link",
    "Claim your free offer today",
    "You are selected for a reward",
    "Limited time offer claim now",
    "Congratulations claim your bonus",
    "You won a special gift",
    "Click this link to win money",
    "Free prize available now",
    "Urgent claim your lottery money",

    "Hi can we meet tomorrow",
    "Please send me the report",
    "Let us discuss the project",
    "Are you coming to class today",
    "The meeting is scheduled for evening",
    "Can you review my assignment",
    "Please call me when free",
    "Let us complete the work",
    "I will send the files later",
    "We have a project discussion today",
    "Please check the document",
    "Can we talk after class",
    "The report is ready",
    "I will attend the meeting",
    "Let us study together tomorrow"
]

y_task_labels = np.array([
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0
])



MAX_WORDS = 2000
MAX_LEN = 20
EMBEDDING_DIM = 64
LSTM_UNITS = 128
BATCH_SIZE = 4


# Create One Common Tokenizer
# Same tokenizer is used for X task and Y task.
# This avoids vocabulary mismatch during fine-tuning.

all_texts = x_task_texts + y_task_texts

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(all_texts)

VOCAB_SIZE = min(MAX_WORDS, len(tokenizer.word_index) + 1)

print("Vocabulary size:", VOCAB_SIZE)

with open("common_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)



def prepare_data(texts, labels):
    sequences = tokenizer.texts_to_sequences(texts)

    padded = pad_sequences(
        sequences,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    return padded, np.array(labels)


def split_data(x_data, y_data, test_ratio=0.3):
    indices = np.arange(len(x_data))
    np.random.shuffle(indices)

    x_data = x_data[indices]
    y_data = y_data[indices]

    test_size = int(len(x_data) * test_ratio)

    x_test = x_data[:test_size]
    y_test = y_data[:test_size]

    x_train = x_data[test_size:]
    y_train = y_data[test_size:]

    return x_train, x_test, y_train, y_test


def build_lstm_model(output_layer_name):
    input_layer = Input(shape=(MAX_LEN,), name="input_layer")

    embedding_layer = Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        name="embedding_layer"
    )(input_layer)

    lstm_layer = LSTM(
        LSTM_UNITS,
        name="lstm_layer"
    )(embedding_layer)

    dropout_layer = Dropout(
        0.3,
        name="dropout_layer"
    )(lstm_layer)

    output_layer = Dense(
        1,
        activation="sigmoid",
        name=output_layer_name
    )(dropout_layer)

    model = Model(
        inputs=input_layer,
        outputs=output_layer
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def print_prediction(label_name, probability):
    if probability >= 0.5:
        return f"{label_name}=1"
    else:
        return f"{label_name}=0"



x_data, x_labels = prepare_data(x_task_texts, x_task_labels)

x_train, x_test, y_x_train, y_x_test = split_data(
    x_data,
    x_labels,
    test_ratio=0.3
)


y_data, y_labels = prepare_data(y_task_texts, y_task_labels)

y_train, y_test, y_y_train, y_y_test = split_data(
    y_data,
    y_labels,
    test_ratio=0.3
)




print("STEP 1: Training X Model on X Task")
print("Task X = Sentiment Classification")


x_model = build_lstm_model(output_layer_name="x_sentiment_output")

x_model.summary()

x_model.fit(
    x_train,
    y_x_train,
    epochs=20,
    batch_size=BATCH_SIZE,
    validation_data=(x_test, y_x_test),
    verbose=1
)

x_loss, x_accuracy = x_model.evaluate(x_test, y_x_test, verbose=0)

print("\nX Model Accuracy on X Task:", x_accuracy)

x_model.save("x_task_sentiment_model.keras")

print("\nX model saved as x_task_sentiment_model.keras")


# . Train Y Model on Y Task from Scratch


print("STEP 2: Training Y Model on Y Task from Scratch")
print("Task Y = Spam Classification")

y_model = build_lstm_model(output_layer_name="y_spam_output")

y_model.summary()

y_model.fit(
    y_train,
    y_y_train,
    epochs=20,
    batch_size=BATCH_SIZE,
    validation_data=(y_test, y_y_test),
    verbose=1
)

y_loss, y_accuracy = y_model.evaluate(y_test, y_y_test, verbose=0)

print("\nY Model Accuracy on Y Task:", y_accuracy)

y_model.save("y_task_spam_model.keras")

print("\nY model saved as y_task_spam_model.keras")


#  Load X Model and Fine-Tune it on Y Task

print("STEP 3: Loading X Model and Fine-Tuning on Y Task")
print("X Model → Fine-Tuned for Spam Classification")

loaded_x_model = load_model("x_task_sentiment_model.keras")


# Remove old X task output and create new Y task output
# We reuse embedding_layer and lstm_layer from X model.

lstm_output = loaded_x_model.get_layer("lstm_layer").output

fine_tune_dropout = Dropout(
    0.3,
    name="fine_tune_dropout"
)(lstm_output)

fine_tuned_y_output = Dense(
    1,
    activation="sigmoid",
    name="fine_tuned_y_spam_output"
)(fine_tune_dropout)

fine_tuned_x_model = Model(
    inputs=loaded_x_model.input,
    outputs=fine_tuned_y_output,
    name="x_model_finetuned_for_y_task"
)


#  First Freeze X Model's Old Layers
# Train only new Y output head

fine_tuned_x_model.get_layer("embedding_layer").trainable = False
fine_tuned_x_model.get_layer("lstm_layer").trainable = False

fine_tuned_x_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining only new Y task output head...\n")

fine_tuned_x_model.fit(
    y_train,
    y_y_train,
    epochs=10,
    batch_size=BATCH_SIZE,
    validation_data=(y_test, y_y_test),
    verbose=1
)


#  Unfreeze X Model Layers and Fine-Tune on Y Task

fine_tuned_x_model.get_layer("embedding_layer").trainable = True
fine_tuned_x_model.get_layer("lstm_layer").trainable = True

fine_tuned_x_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nFine-tuning full X model on Y task...\n")

fine_tuned_x_model.fit(
    y_train,
    y_y_train,
    epochs=10,
    batch_size=BATCH_SIZE,
    validation_data=(y_test, y_y_test),
    verbose=1
)

fine_tuned_loss, fine_tuned_accuracy = fine_tuned_x_model.evaluate(
    y_test,
    y_y_test,
    verbose=0
)

print("\nFine-Tuned X Model Accuracy on Y Task:", fine_tuned_accuracy)

fine_tuned_x_model.save("x_model_finetuned_for_y_task.keras")

print("\nFine-tuned X model saved as x_model_finetuned_for_y_task.keras")



#  Compare Y Model Output vs Fine-Tuned X Model Output


print("STEP 4: Comparing Outputs on Y Task Test Data")


y_model_predictions = y_model.predict(y_test, verbose=0)
fine_tuned_x_predictions = fine_tuned_x_model.predict(y_test, verbose=0)

print("Actual Label | Y Model Output | Fine-Tuned X Output")
print("----------------------------------------------------")

for i in range(len(y_test)):
    actual = y_y_test[i]

    y_prob = y_model_predictions[i][0]
    ft_prob = fine_tuned_x_predictions[i][0]

    y_pred = 1 if y_prob >= 0.5 else 0
    ft_pred = 1 if ft_prob >= 0.5 else 0

    print(
        f"{actual:^12} | "
        f"{y_prob:.4f} -> {y_pred:^5} | "
        f"{ft_prob:.4f} -> {ft_pred:^5}"
    )


# Final Accuracy Comparison



print("FINAL COMPARISON")


print("Y Model trained from scratch on Y Task Accuracy:")
print(y_accuracy)

print("\nX Model fine-tuned on Y Task Accuracy:")
print(fine_tuned_accuracy)

if fine_tuned_accuracy > y_accuracy:
    print("\nResult: Fine-tuned X model performed better on Y task.")
elif fine_tuned_accuracy < y_accuracy:
    print("\nResult: Y model trained from scratch performed better on Y task.")
else:
    print("\nResult: Both models performed equally on Y task.")



# 16. Test with Custom Y Task Sentences

def compare_custom_text(text):
    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    y_model_prob = y_model.predict(padded, verbose=0)[0][0]
    fine_tuned_prob = fine_tuned_x_model.predict(padded, verbose=0)[0][0]

    y_model_class = 1 if y_model_prob >= 0.5 else 0
    fine_tuned_class = 1 if fine_tuned_prob >= 0.5 else 0

    print("\nText:", text)

    print(
        "Y Model Output:",
        y_model_prob,
        "Class:",
        "Spam" if y_model_class == 1 else "Not Spam"
    )

    print(
        "Fine-Tuned X Model Output:",
        fine_tuned_prob,
        "Class:",
        "Spam" if fine_tuned_class == 1 else "Not Spam"
    )


compare_custom_text("Congratulations you won free money")
compare_custom_text("Please send me the project report")
compare_custom_text("Claim your free lottery prize now")
compare_custom_text("Can we meet tomorrow for discussion")

Vocabulary size: 126
STEP 1: Training X Model on X Task
Task X = Sentiment Classification


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Embedding)     │ (None, 20, 64)         │         8,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_layer (LSTM)               │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ x_sentiment_output (Dense)      │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 107,009 (418.00 KB)

 Trainable params: 107,009 (418.00 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Embedding)     │ (None, 20, 64)         │         8,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_layer (LSTM)               │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ y_spam_output (Dense)           │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 107,009 (418.00 KB)

 Trainable params: 107,009 (418.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.5238 - loss: 0.6925 - val_accuracy: 0.3333 - val_loss: 0.7170
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5714 - loss: 0.6853 - val_accuracy: 0.3333 - val_loss: 0.7492
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5714 - loss: 0.6834 - val_accuracy: 0.3333 - val_loss: 0.7919
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5714 - loss: 0.6992 - val_accuracy: 0.3333 - val_loss: 0.8167
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5714 - loss: 0.6808 - val_accuracy: 0.3333 - val_loss: 0.8285
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5714 - loss: 0.6339 - val_accuracy: 0.3333 - val_loss: 0.8312
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7143 - loss: 0.4262 - val_accuracy: 1.0000 - val_loss: 0.4238
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9524 - loss: 0.1854 - val_accuracy: 0.8889 - val_loss: 0.3182
